# PII Handling with Microsoft Presidio — Concepts & Demo

This notebook teaches **how Presidio works** from the ground up, and shows how those concepts add up to the redaction logic used by this app ([`pii_handler.py`](pii_handler.py)).

Presidio has two independent engines:

| Engine | Job | Input | Output |
|---|---|---|---|
| **`AnalyzerEngine`** | *Find* PII | raw text | list of spans (type, position, score) |
| **`AnonymizerEngine`** | *Transform* PII | text + spans | de-identified text |

Analysis and anonymization are deliberately decoupled: the analyzer decides *what* is PII, and *operators* on the anonymizer decide *what to do about it* (mask, replace, hash, encrypt...).

We'll cover, in order:
1. Analyzer basics — the `RecognizerResult`
2. Built-in recognizers & entity types
3. Confidence scores and overlapping matches
4. Anonymizer operators — replace / mask / redact / hash
5. Custom recognizers — `PatternRecognizer`, context words, deny-lists, validation
6. Consistent tokens via a custom operator (what this app does)
7. Reversible anonymization — encrypt + `DeanonymizeEngine`
8. How it all maps back to `pii_handler.py`

![Diagram](./images/01_img.png)

In [1]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine, DeanonymizeEngine
from presidio_anonymizer.entities import OperatorConfig
from presidio_anonymizer.operators import Operator, OperatorType

# Building the analyzer loads the spaCy model, so do it once and reuse it.
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()
print("engines ready")

engines ready


## 1. Analyzer basics — the `RecognizerResult`

`analyzer.analyze(text, language)` returns a list of **`RecognizerResult`** objects. Each one is just four things:

- `entity_type` — e.g. `PERSON`, `EMAIL_ADDRESS`
- `start`, `end` — character offsets into the text (the actual value is `text[start:end]`)
- `score` — confidence from 0.0 to 1.0

Note it returns *positions*, not the substrings — that's what lets the anonymizer splice replacements in later.

In [2]:
text = "Hi, I'm John Smith. Email me at john.smith@example.com or call 415-555-1234."

results = analyzer.analyze(text=text, language="en")

for r in sorted(results, key=lambda r: r.start):
    print(f"{r.entity_type:<15} score={r.score:<4} [{r.start:>2}:{r.end:<2}]  ->  {text[r.start:r.end]!r}")

PERSON          score=0.85 [ 8:18]  ->  'John Smith'
EMAIL_ADDRESS   score=1.0  [32:54]  ->  'john.smith@example.com'
URL             score=0.5  [32:39]  ->  'john.sm'
URL             score=0.5  [43:54]  ->  'example.com'
PHONE_NUMBER    score=0.4  [63:75]  ->  '415-555-1234'


## 2. Built-in recognizers & entity types

Out of the box Presidio ships recognizers for many common entities. Two kinds:

- **NER-based** (`PERSON`, `LOCATION`, `NRP`...) — from the spaCy statistical model. Great recall on names/places *in its training distribution*, patchy outside it.
- **Pattern/checksum-based** (`EMAIL_ADDRESS`, `CREDIT_CARD`, `US_SSN`, `IBAN_CODE`, `IP_ADDRESS`...) — regex + validation (e.g. credit cards are Luhn-checked).

You can list what the current registry supports:

In [5]:
len(sorted(analyzer.get_supported_entities()))

19

You can also restrict analysis to only the entities you care about with the `entities=` argument — faster, and avoids surprising matches. This app only cares about a fixed set:

In [6]:
WANTED = ["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "CREDIT_CARD", "US_SSN", "IP_ADDRESS"]

sample = "Jane pays with card 4111 1111 1111 1111 from IP 10.0.0.5; SSN 123-45-6789."
for r in sorted(analyzer.analyze(text=sample, language="en", entities=WANTED), key=lambda r: r.start):
    print(f"{r.entity_type:<15} score={r.score:<4}  ->  {sample[r.start:r.end]!r}")

PERSON          score=0.85  ->  'Jane'
CREDIT_CARD     score=1.0   ->  '4111 1111 1111 1111'
IP_ADDRESS      score=0.95  ->  '10.0.0.5'


## 3. Confidence scores and overlapping matches

This is the subtle part. **Multiple recognizers can claim the same (or overlapping) span**, each with its own score. A 10-digit number might be flagged as `PHONE_NUMBER` *and* `US_BANK_NUMBER` *and* `US_DRIVER_LICENSE` at different confidences.

If you naively splice every result into the text, overlapping spans corrupt the output. So you must **resolve overlaps** first — typically keep the highest-scoring span and drop anything overlapping it.

Watch how many entities compete for the same digits:

In [7]:
ambiguous = "my number is 4155551234"
raw = analyzer.analyze(text=ambiguous, language="en")

print("all raw candidates (note the overlaps and score spread):")
for r in sorted(raw, key=lambda r: (-r.score)):
    print(f"  {r.entity_type:<18} score={r.score:<5} [{r.start}:{r.end}] {ambiguous[r.start:r.end]!r}")

# Greedy overlap resolution: highest score first, drop anything overlapping a kept span.
kept = []
for r in sorted(raw, key=lambda r: (-r.score, -(r.end - r.start))):
    if not any(r.start < k.end and k.start < r.end for k in kept):
        kept.append(r)

print("\nafter overlap resolution:")
for r in kept:
    print(f"  {r.entity_type:<18} score={r.score} {ambiguous[r.start:r.end]!r}")

all raw candidates (note the overlaps and score spread):
  PHONE_NUMBER       score=0.75  [13:23] '4155551234'
  US_BANK_NUMBER     score=0.05  [13:23] '4155551234'
  US_DRIVER_LICENSE  score=0.01  [13:23] '4155551234'

after overlap resolution:
  PHONE_NUMBER       score=0.75 '4155551234'


> Good news: the `AnonymizerEngine` in the next section handles overlap resolution for you. The manual version above is worth understanding because `pii_handler.py` does its own splicing (to build stable tokens), so it has to resolve overlaps itself — that's the `by_priority` / `kept` loop in `redact()`.

## 4. Anonymizer operators — replace / mask / redact / hash

Once you have analyzer results, hand them to `anonymizer.anonymize(...)` with a dict of **operators**. An operator says *how* to transform a given entity type. Built-ins:

- **`replace`** — swap in a fixed string (default: `<ENTITY_TYPE>`)
- **`mask`** — keep some characters, hide the rest with a char (great for "last 4 of card")
- **`redact`** — delete the value entirely
- **`hash`** — one-way hash (sha256 etc.)
- **`keep`** — leave it untouched

The special key `DEFAULT` applies to any entity you didn't name explicitly.

In [8]:
text = "John Smith, john@x.com, card 4111111111111111, phone 4155551234"
res = analyzer.analyze(text=text, language="en",
                       entities=["PERSON", "EMAIL_ADDRESS", "CREDIT_CARD", "PHONE_NUMBER"])

out = anonymizer.anonymize(
    text=text,
    analyzer_results=res,
    operators={
        "DEFAULT":       OperatorConfig("replace", {"new_value": "<REDACTED>"}),
        "PERSON":        OperatorConfig("replace", {"new_value": "<PATIENT>"}),
        "EMAIL_ADDRESS": OperatorConfig("redact"),
        "CREDIT_CARD":   OperatorConfig("mask", {"masking_char": "*",
                                                  "chars_to_mask": 12,
                                                  "from_end": False}),
        "PHONE_NUMBER":  OperatorConfig("hash", {"hash_type": "sha256"}),
    },
)

print("IN: ", text)
print("OUT:", out.text)

IN:  John Smith, john@x.com, card 4111111111111111, phone 4155551234
OUT: <PATIENT>, , card ************1111, phone 15409792d650c04f53e60cbfb7743f0d6d672644da514a8380ff6c8636c40d56


## 5. Custom recognizers

The built-in NER model misses things — especially in a narrow conversational domain. This app's medical intake flow has three gaps the defaults don't cover well:

1. **Age** (`"34"`, `"aged 42"`, `"25 years old"`) — not a default entity at all
2. **Sex/gender** keywords
3. **Names outside the NER training distribution** (many South Asian / Arabic given names return *zero* candidates)

The fix is a **`PatternRecognizer`**: a supported entity + a list of `Pattern(name, regex, score)`. You register it on the analyzer and it runs alongside the built-ins.

In [9]:
import re

age_recognizer = PatternRecognizer(
    supported_entity="AGE",
    patterns=[
        Pattern("age_years_old", r"\b\d{1,3}\s*(?:years?\s*old|yr\.?\s*old|y\.?o\.?)\b", 0.9),
        Pattern("age_context",   r"\b(?:age[d]?|am|i'm|i\s+am)\s*(?:is|:|-)?\s+\d{1,3}\b", 0.85),
        Pattern("age_standalone", r"^\s*\d{1,3}\s*$", 0.6),   # bare "34" as a whole answer
    ],
    global_regex_flags=re.IGNORECASE,
)

# A fresh analyzer so we can see the effect of adding our recognizer.
custom = AnalyzerEngine()
custom.registry.add_recognizer(age_recognizer)

for t in ["I am 42 years old", "aged 30", "34", "the year 1999"]:
    hits = custom.analyze(text=t, language="en", entities=["AGE"])
    print(f"{t!r:<22} -> {[(t[h.start:h.end], round(h.score,2)) for h in hits]}")

'I am 42 years old'    -> [('42 years old', 0.9), ('I am 42', 0.85)]
'aged 30'              -> [('aged 30', 0.85)]
'34'                   -> [('34', 0.6)]
'the year 1999'        -> []


### Deny-lists and `validate_result` — cutting false positives

A loose "any Title-Case word is a name" fallback catches unknown names, but it *also* catches `"Cardiology"`, `"Yes"`, `"Male"`, etc. Two tools to tighten a recognizer:

- **`deny_list`** — force these exact words to match an entity (the opposite problem)
- **`validate_result(pattern_text)`** — a hook returning `True` (accept), `False` (reject), or `None` (no opinion). This app uses it to *reject* known non-name words.

Also useful: **`context`** words — nearby tokens that *boost* a match's score (e.g. the word "phone" near a number).

In [10]:
NON_NAME_WORDS = {"yes", "no", "male", "female", "cardiology", "neurology"}

name_recognizer = PatternRecognizer(
    supported_entity="PERSON",
    patterns=[Pattern("name_fallback", r"^[A-Z][a-zA-Z'-]*(?:\s+[A-Z][a-zA-Z'-]*){0,3}$", 0.6)],
)
# Reject known non-names; return None (abstain) for everything else.
name_recognizer.validate_result = (
    lambda text: False if text.strip().lower() in NON_NAME_WORDS else None
)

custom.registry.add_recognizer(name_recognizer)

for t in ["Rajesh Kumar", "Cardiology", "Yes", "Priya"]:
    hits = custom.analyze(text=t, language="en", entities=["PERSON"])
    verdict = "NAME" if hits else "(not a name)"
    print(f"{t!r:<16} -> {verdict}")

'Rajesh Kumar'   -> NAME
'Cardiology'     -> (not a name)
'Yes'            -> (not a name)
'Priya'          -> NAME


## 6. Consistent tokens via a custom operator

`replace` gives every `PERSON` the *same* placeholder — you lose track of which is which, and can't restore them later. For a conversation you usually want **stable, numbered tokens**: the same original value always maps to the same token (`John Smith` → `<PERSON_0>` every time), and different values get different numbers.

The Presidio-native way is a **custom operator** — subclass `Operator`, keep a per-value counter/map, and register it with `add_anonymizer`. This is exactly the behaviour `pii_handler.py` implements (it just does the splicing by hand instead of via an operator).

In [11]:
class InstanceCounterAnonymizer(Operator):
    """Map each distinct original value to a stable, numbered token per entity type."""

    def operate(self, text, params=None):
        entity_type = params["entity_type"]
        mapping = params["entity_mapping"]          # {entity_type: {original: token}}
        per_type = mapping.setdefault(entity_type, {})

        if text in per_type:                        # seen this exact value -> reuse token
            return per_type[text]
        token = f"<{entity_type}_{len(per_type)}>"
        per_type[text] = token
        return token

    def validate(self, params=None):
        pass

    def operator_name(self):
        return "entity_counter"

    def operator_type(self):
        return OperatorType.Anonymize


counting_anonymizer = AnonymizerEngine()
counting_anonymizer.add_anonymizer(InstanceCounterAnonymizer)

entity_mapping = {}   # persists across calls -> stable tokens across conversation turns
turns = [
    "John Smith met Jane Doe.",
    "John Smith called again.",     # same person -> same token as turn 1
]
for t in turns:
    r = analyzer.analyze(text=t, language="en", entities=["PERSON"])
    out = counting_anonymizer.anonymize(
        text=t, analyzer_results=r,
        operators={"DEFAULT": OperatorConfig("entity_counter", {"entity_mapping": entity_mapping})},
    )
    print(f"{t!r:<28} -> {out.text}")

print("\nmap:", entity_mapping)

'John Smith met Jane Doe.'   -> <PERSON_1> met <PERSON_0>.
'John Smith called again.'   -> <PERSON_1> called again.

map: {'PERSON': {'Jane Doe': '<PERSON_0>', 'John Smith': '<PERSON_1>'}}


## 7. Reversible anonymization — encrypt & `DeanonymizeEngine`

Sometimes you need to get the original values *back* (e.g. show the real name to the clinician, but never store or log it). Two approaches:

1. **Keep the mapping** from section 6 and just reverse it (what this app does — display-only).
2. **Encrypt** with a key, then **decrypt** with `DeanonymizeEngine`. No mapping to store — the ciphertext *is* the record — but anyone with the key can reverse it.

Here's the encrypt/decrypt round-trip:

In [12]:
KEY = "WmZq4t7w!z%C&F)J"   # 16 chars = AES-128; in real code load from a secret store

text = "Patient John Smith, email john@x.com"
res = analyzer.analyze(text=text, language="en", entities=["PERSON", "EMAIL_ADDRESS"])

encrypted = anonymizer.anonymize(
    text=text, analyzer_results=res,
    operators={"DEFAULT": OperatorConfig("encrypt", {"key": KEY})},
)
print("ENCRYPTED:", encrypted.text)

# encrypted.items carries the spans of the ciphertext we just produced
deanonymizer = DeanonymizeEngine()
decrypted = deanonymizer.deanonymize(
    text=encrypted.text,
    entities=encrypted.items,
    operators={"DEFAULT": OperatorConfig("decrypt", {"key": KEY})},
)
print("DECRYPTED:", decrypted.text)

ENCRYPTED: Patient 27rM8pw4Z8Ce_zmTj2wxgEC8Zcpaw1cw8CEx_Ze_tyQ=, email vxs23wVT5uzCwwpdUME7usL8C3vUxLsxO4JJ7Z5moGA=
DECRYPTED: Patient John Smith, email john@x.com


## 8. How this maps back to `pii_handler.py`

Everything above is what the app's [`PIIHandler`](pii_handler.py) stitches together:

| Concept in this notebook | Where it lives in `pii_handler.py` |
|---|---|
| `AnalyzerEngine` + `entities=` filter | `__init__` / `ENTITIES` list |
| Custom `AGE`, `SEX`, phone, name recognizers | the `PatternRecognizer`s in `__init__` |
| `validate_result` to drop non-names | `name_recognizer.validate_result` |
| Overlap resolution (§3) | the `by_priority` / `kept` loop in `redact()` |
| Stable numbered tokens (§6) | `_entity_map` + `_counters`, `TOKEN_<TYPE>_<n>` |
| Reversible via kept mapping (§7, option 1) | `deredact()` / `deredact_info()` |

The app rolls its own splicing + token map (rather than using a custom operator) so it has one dict it can both redact with and reverse for display — but conceptually it's doing exactly what sections 3, 5, and 6 describe.

To see the finished handler in action, you can still import it directly:

```python
from pii_handler import PIIHandler
pii = PIIHandler()
pii.redact("My name is John Smith, I'm 34, reach me at 9999888877")
```

## Playground — try your own text

Edit `my_text` and re-run. Uses the analyzer with the custom AGE/name recognizers registered in section 5.

In [11]:
my_text = "Type anything with a name, phone, email, SSN, or age here..."

results = custom.analyze(text=my_text, language="en")
print("detected:")
for r in sorted(results, key=lambda r: r.start):
    print(f"  {r.entity_type:<15} {my_text[r.start:r.end]!r} (score={round(r.score,2)})")

out = anonymizer.anonymize(
    text=my_text, analyzer_results=results,
    operators={"DEFAULT": OperatorConfig("replace", {"new_value": "<HIDDEN>"})},
)
print("\nanonymized:", out.text)

detected:

anonymized: Type anything with a name, phone, email, SSN, or age here...
